# Исследования пользователей оператора сотовой связи

Вы аналитик оператора сотовой связи. Клиентам предлагают два тарифных плана: «Смарт» и «Ультра». Чтобы скорректировать рекламный бюджет, коммерческий департамент хочет понять, какой тариф приносит больше денег.
Вам предстоит сделать предварительный анализ тарифов на небольшой выборке клиентов. В вашем распоряжении данные 500 пользователей за 2018 год. Нужно проанализировать поведение клиентов и сделать вывод — какой тариф лучше.

Описание тарифов

Тариф «Смарт»

1.	Ежемесячная плата: 550 рублей

2.	Включено 500 минут разговора, 50 сообщений и 15 Гб интернет-трафика

3.	Стоимость услуг сверх тарифного пакета:

o	минута разговора: 3 рубля

o	сообщение: 3 рубля

o	1 Гб интернет-трафика: 200 рублей

Тариф «Ультра»

1.	Ежемесячная плата: 1950 рублей

2.	Включено 3000 минут разговора, 1000 сообщений и 30 Гб интернет-трафика

3.	Стоимость услуг сверх тарифного пакета:

o	минута разговора: 1 рубль

o	сообщение: 1 рубль

o	1 Гб интернет-трафика: 150 рублей

Примечание:

Оператор всегда округляет секунды до минут, а мегабайты — до гигабайт. Каждый звонок округляется отдельно: даже если он длился всего 1 секунду, будет засчитан как 1 минута.
Для веб-трафика отдельные сессии не считаются. Вместо этого общая сумма за месяц округляется в бо́льшую сторону. Если абонент использует 1025 мегабайт в этом месяце, с него возьмут плату за 2 гигабайта.

## Описание данных

Таблица users (информация о пользователях):

  *	user_id — уникальный идентификатор пользователя

  *	first_name — имя пользователя

  *	last_name — фамилия пользователя

  *	age — возраст пользователя (годы)

  *	reg_date — дата подключения тарифа (день, месяц, год)

  *	churn_date — дата прекращения пользования тарифом (если значение пропущено, то тариф ещё действовал на момент выгрузки данных)
  *	city — город проживания пользователя

  *	tarif — название тарифного плана

Таблица calls (информация о звонках):

  *	id — уникальный номер звонка

  *	call_date — дата звонка

  *	duration — длительность звонка в минутах

  *	user_id — идентификатор пользователя, сделавшего звонок


Таблица messages (информация о сообщениях):

  *	id — уникальный номер сообщения
  
  *	message_date — дата сообщения
  
  *	user_id — идентификатор пользователя, отправившего сообщение


Таблица internet (информация об интернет-сессиях):

  *	id — уникальный номер сессии

  *	mb_used — объём потраченного за сессию интернет-трафика (в мегабайтах)

  *	session_date — дата интернет-сессии

  *	user_id — идентификатор пользователя


Таблица tariffs (информация о тарифах):
  
  *	tariff_name — название тарифа
  
  *	rub_monthly_fee — ежемесячная абонентская плата в рублях
  
  *	minutes_included — количество минут разговора в месяц, включённых в абонентскую плату

  *	messages_included — количество сообщений в месяц, включённых в абонентскую плату

  *	mb_per_month_included — объём интернет-трафика, включённого в абонентскую плату (в мегабайтах)

  *	rub_per_minute — стоимость минуты разговора сверх тарифного пакета (например, если в тарифе 100 минут разговора в месяц, то со 101 минуты будет взиматься плата)

  *	rub_per_message — стоимость отправки сообщения сверх тарифного пакета

  *	rub_per_gb — стоимость дополнительного гигабайта интернет-трафика сверх тарифного пакета (1 гигабайт = 1024 мегабайта)

# Шаг 1. Откройте файлы с данными и изучите общую информацию

In [3]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as st

Загрузите файлы `users.csv`,  `tariffs.csv`, `calls.csv`, `internet.csv`, `messages.csv`

In [ ]:
# загрузите файл с диска, из директории или из временной папки
try:

  users = pd.read_csv('users.csv', sep=",")
  tariffs = pd.read_csv('tariffs.csv', sep=",")
  calls = pd.read_csv('calls.csv', sep=",")
  internet = pd.read_csv('internet.csv', sep=",", index_col=0)
  messages = pd.read_csv('messages.csv', sep=",")
except FileNotFoundError:
  from google.colab import drive
  drive.mount('/content/drive')
  users = pd.read_csv('/content/drive/My Drive/users.csv', sep=",")
  tariffs = pd.read_csv('/content/drive/My Drive/tariffs.csv', sep=",")
  calls = pd.read_csv('/content/drive/My Drive/calls.csv', sep=",")
  internet = pd.read_csv('/content/drive/My Drive/internet.csv', sep=",", index_col=0)
  messages = pd.read_csv('/content/drive/My Drive/messages.csv', sep=",")


Изучите общую информацию о данных, содержащихся в файлах. Изучите типы данных. Оцените число пропусков в каждом датасете.

In [ ]:
for name in [users, calls, internet, messages, tariffs]:
  print('dataset')
  display(name.info())
  print('Количество пропусков:')
  display(name.isnull().sum())




Датасет `users` содержит информацию о пользователях. Столбец `churn_date` содержит большое количество значений `NaN`, т.к. эти пользователи ещё активны. Столбцы `reg_date` и `churn_date` являются типом `object`, необходимо перевести их в формат даты.

Датасет `calls` содержит сведения о совершенных звонках и их продолжительности. Пропусков нет. Столбец `call_date` является типом `object`, необходимо перевести в формат даты. Столбец `id` характеризует идентификатор записи, т.е. каждый совершенный звонок, совершённый пользоватлем.

Датасет `messages` содержит даты переданных сообщений. Пропусков нет. Столбец `message_date`	 является типом `object`, необходимо перевести в формат даты.

Датасет `internet` содержит информацию об израсходаванных Мб траффика интернета и даты сессий пользователей. Пропусков нет. Столбец `session_date` является типом object, необходимо пеервести в формат даты. Столбец `Unnamed: 0` содержит номер записи, т.е. новую сессию для каждого пользователя. Никакой информации в себе не несет, т.к. есть столбец `id`, характреизующий каждую сессию. Этот столбец можно удалить.

Датасет `tariffs` содержит сведения о тарифах. Пропусков нет.


# Шаг 2. Подготовьте данные

•	Приведите данные к нужным типам;

•	Найдите и исправьте ошибки в данных, если они есть.

Поясните, какие ошибки вы нашли и как их исправили. В данных вы найдёте звонки с нулевой продолжительностью. Это не ошибка: нулями обозначены пропущенные звонки, поэтому их не нужно удалять.

Приведем к формату даты столбцы `reg_date` и `churn_date` в `users`, `call_date` в `calls`, `message_date` в `messages` и `session_date` в `internet` методом `pd.to_datetime`

In [ ]:
users['reg_date'] = pd.to_datetime(users['reg_date'])
users['churn_date'] =
calls['call_date'] =
messages['message_date'] =
internet['session_date'] =

In [ ]:
users.dtypes

Изучите статистические характеристики. Выведите на экран описательную статистику для всех датасетов, кроме `messeges`, т.к. в нем нет количественных переменных (кроме `user_id`). Используйте метод `describe()`

Никаких подозрительных значений (типа отрицательного возраста, отрицательного числа минут разговоров или отрицательных значений использованного траффика не обнаружено).


Проверьте данные на наличие явных дубликатов методом `duplicated()`.

Явных дубликатов нет.  

Проверьте наличие неявных дубликатов в информации о пользователях. Для этого рассмотрите уникальные значения в именах пользователей, городов проживания. Используйте метод `unique()`

In [ ]:
users['city'].value_counts().head(10)

Неверных написаний фамилий, имён пользователей и городов их проживания не обнаружено. Неявных дубликатов в датасете `users` нет.

Неявных дубликатов тоже не обнаружено

В примечании сказано, что «Мегалайн» всегда округляет секунды до минут, а мегабайты — до гигабайт. Каждый звонок округляется отдельно: даже если он длился всего 1 секунду, будет засчитан как 1 минута.

Округлите длительности разговоров каждого пользователя до целого вверх. Используйте метод `ceil()`.

**np.ceil()** — функция библиотеки NumPy, которая округляет элементы массива вверх до ближайшего целого числа (к «потолку»).

Принцип работы: возвращает наименьшее целое число i, такое что i≥x (где x — исходное значение).

*Синтаксис:*

import numpy as np
result = np.ceil(a)

где a — число, список или массив NumPy.

Важные особенности:

Работает поэлементно для массивов любой размерности.

Для отрицательных чисел округление «вверх» означает к нулю: np.ceil(-1.9) → -1.0.

Результат всегда имеет тип с плавающей точкой (float), даже если значение целое: 2.0, а не 2.

NaN и бесконечности (np.nan, np.inf, -np.inf) остаются без изменений.

In [ ]:
calls['duration'] = np.ceil(calls['duration']).astype(int)

### Посчитайте для каждого пользователя:
•	количество сделанных звонков и израсходованных минут разговора по месяцам;

•	количество отправленных сообщений по месяцам;

•	объем израсходованного интернет-трафика по месяцам;

•	помесячную выручку с каждого пользователя (вычтите бесплатный лимит из суммарного количества звонков, сообщений и интернет-трафика; остаток умножьте на значение из тарифного плана; прибавьте абонентскую плату, соответствующую тарифному плану).


In [ ]:
#Выделим месяц совершения звонка в датасете calls и сформируем таблицу calls_month с количеством сделанных звонков
#и израсходованных минут разговора по месяцам

#calls['month'] = pd.DatetimeIndex(calls['call_date']).month
calls['month']=calls['call_date'].dt.month
calls_month = calls.groupby(['user_id', 'month'])['duration'].agg(['count', 'sum']).reset_index()
calls_month.columns = ['user_id', 'month', 'call_per_month', 'minute_per_month']
calls_month.head()

In [ ]:
#Выделим месяц отправки сообщений в датасете messages и сформируем  таблицу messages_month
#с количеством отправленных сообщений по месяцам


In [ ]:
#Выделим месяц использования интернета в датасете internet и сформируем сводную таблицу internet_month
#с объемом израсходованного интернет-трафика по месяцам



В примечании сказано, что для веб-трафика отдельные сессии не считаются. Вместо этого общая сумма за месяц округляется в бо́льшую сторону. Если абонент использует 1025 мегабайт в этом месяце, с него возьмут плату за 2 гигабайта.

Поэтому округлите полученные результаты объема использованного траффика в большую сторону до целого.


In [ ]:
internet_month['mb_per_month']=
internet_month['gb_per_month']=
internet_month.head()

Выясните количество абонентов, которые совершают звонки, отправляют СМС и используют интернет. Используйте метод `nunique()`

In [ ]:
# Определим количество уникальных абонентов, которые совершали звонки, отправляли СМС, использовали интернет-траффик




Видим, что не все абоненты совершали звонки, отправляли сообщения, использовали интернет-траффик

Объединим все полученные таблицы со сведениямии об абонентах в один датафрейм abonents методом `merge()`.

In [ ]:
# объединение calls_month и messages_month
abonents =
# объединение abonents и internet_month
abonents =
# проверка
abonents.head()

Добавим в этот датафрейм следующие сведения о пользователях из таблицы `users`: город, тарифный план.

In [ ]:
# выделим нужные столбцы из таблицы users
user1 = users.loc[1:, []]

# объединим abonents и user1 по столбцу'user_id'

abonents =

abonents.head()

Проверьте количество абонентов в полученной таблице методом `nunique()`

In [ ]:
# Определите количество пропущенных значений в столбцах month, calls_per_month, messages_per_month, mb_per_month в таблице abonents


In [ ]:
# Выясните, есть ли абоненты, которые не совершали звонки, не отправляли СМС, не использовали интернет-траффик


Видим, что два абонента не пользовались ни интернетом, не отправляли сообщений и не совершали звонки. И столбец `month` для этих пользователей пуст.

Проверьте дату подключения этих абонентов. Обратитесь к таблице `users` и проверьте `reg_data` для этих пользователей

Эти абоненты были подключены в декабре, поэтому пропущенное значение в столбце month таблицы abonents замените на значение 12.

Используйте метод fillna()

In [ ]:
abonents['month'] =

Заполните отсутствующие значения в `calls_per_month`, `minute_per_month`, `messages_per_month`, `gb_per_month` значением 0.

Используйте метод `fillna()`

In [ ]:
abonents.head()

In [ ]:
#Измените тип данных на целое в столбцах calls_per_month, minute_per_month, messages_per_month, mb_per_month, gb_per_month
abonents[['month','call_per_month', 'minute_per_month', 'messages_per_month', 'mb_per_month', 'gb_per_month']] = abonents[['month', 'call_per_month', 'minute_per_month', 'messages_per_month','mb_per_month', 'gb_per_month']].astype(int)
abonents.dtypes

Добавьте в таблицу `abonents` сведения о помесячной выручке с каждого пользователя. Для этого обратитесь к таблице `tariffs` с описанием тарифных планов.

• помесячная выручка с каждого пользователя (вычтите бесплатный лимит из суммарного количества звонков, сообщений и интернет-трафика; остаток умножьте на значение из тарифного плана; прибавьте абонентскую плату, соответствующую тарифному плану).

In [ ]:
tariffs

**ВАРИАНТ 1**

ШАГ 1. Присоедим к abonents таблицу tariffs

In [ ]:
abonents_tariffs = abonents.merge (tariffs, left_on='tariff', right_on='tariff_name')
abonents_tariffs.head()

ШАГ 2. Рассчитываем компоненты выручки и добавляем новый столбец в датафрейм abonents

Для замены отрицательных значений используем метод `.clip(lower = 0)`

`clip(lower=0)` — метод в Pandas для ограничения значений в Series или DataFrame: заменяет все значения, которые меньше указанного нижнего порога (lower), на этот порог.

Как работает:

Если значение ≥ 0 → остаётся без изменений.

Если значение < 0 → заменяется на 0.

*Синтаксис:*

**series_or_dataframe.clip(lower=0)**

In [ ]:
inc_calls = (abonents_tariffs['minute_per_month'] - abonents_tariffs['minutes_included']) \
  * abonents_tariffs['rub_per_minute']
inc_calls = inc_calls.clip(lower=0)  # заменяем отрицательные значения на 0

inc_mess =

inc_mess =


# int_r(num) - Функция округления вверх. Описывалась выше
inc_mb  = np.ceil((abonents_tariffs['mb_per_month'] - abonents_tariffs['mb_per_month_included'])/1024) * \
abonents_tariffs['rub_per_gb']

inc_mb =

abonents['income'] =

abonents.head()

ВАРИАНТ 2. **ИСПОЛЬЗОВАНИЕ СОБСТВЕННОЙ** **ФУНКЦИИ**

In [ ]:
def income_month(abonents):

    if abonents['tariff'] == 'smart':

        inc_calls = (abonents['minute_per_month'] - tariffs.loc[0,'minutes_included']) * tariffs.loc[0,'rub_per_minute']
        if inc_calls < 0 :
            inc_calls = 0

        inc_mess = (abonents['messages_per_month'] - tariffs.loc[0,'messages_included']) * tariffs.loc[0,'rub_per_message']
        if inc_mess < 0 :
            inc_mess = 0

        # int_r(num) - Функция округления вверх. Описывалась выше
        inc_mb  = np.ceil((abonents['mb_per_month'] - tariffs.loc[0,'mb_per_month_included'])/1024) * tariffs.loc[0,'rub_per_gb']
        if inc_mb < 0 :
            inc_mb = 0

        inc = inc_calls + inc_mess + inc_mb + tariffs.loc[0,'rub_monthly_fee']

    else:

        inc_calls = (abonents['minute_per_month'] - tariffs.loc[1,'minutes_included']) * tariffs.loc[1,'rub_per_minute']
        if inc_calls < 0 :
            inc_calls = 0

        inc_mess = (abonents['messages_per_month'] - tariffs.loc[1,'messages_included']) * tariffs.loc[1,'rub_per_message']
        if inc_mess < 0 :
            inc_mess = 0

        inc_mb  = np.ceil((abonents['mb_per_month'] - tariffs.loc[1,'mb_per_month_included'])/1024) * tariffs.loc[1,'rub_per_gb']
        if inc_mb < 0 :
            inc_mb = 0

        inc = inc_calls + inc_mess + inc_mb + tariffs.loc[1,'rub_monthly_fee']

    return inc



In [ ]:
abonents['income'] = abonents.apply(income_month, axis=1)
abonents.head()

### Вывод
В данных не обнаружено пропусков и дубликатов, значения NaN соответствуют нулевым значениям, замена на которые была выполнена. Сформирована новая таблица, содержащая сведения об израсходаванных минутах, сообщениях и Мб интернет-траффика для каждого пользователя ежемесячно.

Не все абоненты совершали звонки, отправляли сообщения, использовали интернет-траффик.

Определена ежемесячная выручка от каждого абонента с учетом тарифа подключения.

# Шаг 3. Проанализируйте данные

Опишите поведение клиентов оператора, исходя из выборки. Сколько минут разговора, сколько сообщений и какой объём интернет-трафика требуется пользователям каждого тарифа в месяц? Посчитайте среднее количество, дисперсию и стандартное отклонение. Постройте гистограммы. Опишите распределения.

Определите количество абонентов, подключенных к различным тарифам. Постройте график в виде столбчатой диаграммы методом `countplot()`.


`Countplot` — это тип графика в библиотеке Seaborn (Python), который визуализирует частоту (количество) наблюдений для каждой категории в категориальной переменной. По сути, это столбчатая диаграмма, где высота столбца показывает, сколько раз данная категория встречается в данных.

*Ключевые особенности*

Автоматически считает количество значений в каждой категории — не нужно предварительно агрегировать данные.

Подходит для номинальных и порядковых данных (пол, день недели, тип товара и т. д.).

Интегрирован в экосистему Seaborn — легко настраивать стиль и цвета.

Поддерживает группировку по дополнительной категории через параметр hue.

Может быть вертикальным (x) или горизонтальным (y).

Базовый синтаксис
python
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(
    
    data=df,           # DataFrame с данными
    
    x='column_name',   # категориальная переменная для оси X (вертикальные столбцы)
    
    y='column_name',   # или для оси Y (горизонтальные столбцы)
    
    hue='group_col',  # дополнительная группировка (например, по полу)
    
    palette='Set2',     # цветовая палитра
    
    order=['A', 'B', 'C']  # порядок категорий
)
plt.show()

69,3% абонентов подключены к тарифу "smart", остальные 30,7% - к тарифу "ultra"

In [ ]:
# разобьем всех абонентов на 2 таблицы по тарифу: smart_abonent и ultra_abonent
smart_abonent =
ultra_abonent =

display(smart_abonent.head())
display(ultra_abonent.head())

In [ ]:
# сгрупируем данные об абонентах различных тарифов по месяцам и вычислим суммарные показатели за месяц
smart_ab_month =
ultra_ab_month =

In [ ]:
# Построим графики количества минут разговоров, отправленных сообщений, объема интернет-траффика всех абонентов каждого тарифного плана по месяцам
for x in ['minute_per_month', 'messages_per_month', 'mb_per_month','income']:
  plt.figure(figsize = (9,6))
  plt.bar(smart_ab_month['month']-0.2, smart_ab_month[x], label = 'smart', color = 'r', alpha = 0.5, width = 0.4)
  plt.bar()

  plt.xlabel('month')
  plt.ylabel(x)
  plt.grid()
  plt.legend(loc = 'best')
  plt.show()



Из графиков видим, что суммарное количество минут разговоров, отправленных сообщений, интернет-траффика и выручки от абонентов тарифа "smart" превышает аналогичные показатели для абонентов тарифа "ulrta".

Общее количество минут разговора, сообщений и объём интернет-трафика пользователям тарифа "smart" в месяц требуется больше, чем абонентам тарифа "ultra".  Но нужно учитывать, что число абонентов тарифа "smart" более, чем в два раза превышает число абонентов тарифа "ultra".

Поэтому построим аналогичные графики для средних показателей на одного абонента.


In [ ]:
smart_ab_month_mean =
ultra_ab_month_mean =

In [ ]:
# Построим графики количества минут разговоров, отправленных сообщений, объема интернет-траффика на одного абонентов каждого тарифного плана по месяцам
for x in ['minute_per_month', 'messages_per_month', 'mb_per_month','income']:
  plt.figure(figsize = (9,6))
  plt.bar()
  plt.bar()

  # Добавим горизонтальную линию - количество единиц, включенных в тариф "smart"
  if x == 'minute_per_month':
    porog = 500
  elif x == 'messages_per_month':
    porog = 50
  elif x == 'mb_per_month':
    porog = 15360
  else:
    porog = 550

  plt.axhline(y = porog, label = 'пороговое значение для smart', color = 'g')

  plt.xlabel('month')
  plt.ylabel(x)
  plt.grid()
  plt.legend(loc = 'best')
  plt.show()

Видим другую картину. Во все месяцы, кроме февраля, число израсходованных минут разговоров, отправленных сообщений и израсходованного интернет-траффика у абонентов тарифного плана "smart" в среднем было меньше, чем у абонентов тарифа "ultra". И выручка, полученная со всех абонентов в месяц в среднем была выше на тарифном плане "ultra" ежемесячно.

Из графиков видим, что абоненты тарифа smart не полностью используют пакет минут и сообщений тарифного плана. С мая (5 месяца) эти абоненты расходуют интернет-траффика немного больше, чем предусмотрено абонентной платой. То есть основную часть выручки формирует оплата интернет-траффика сверх тарифного плана.

Для пользователей тарифа ultra пороговые значения, предусмотренные тарифным планом, тоже не расходуются в полном объеме.

Можем сказать, что в среднем в месяц каждый абонент тарифа ultra в месяц совершает больше звонков, отправляет больше сообщений, использует больший объем интернет-траффика, чем абонент тарифного плана smart. Но это всё в рамках общего пакета ежемесячной платы.

Оценим средние показатели. Вычислим основные числовые характеристики случайных величин числа израсходованных минут разговоров, отправленных сообщений и израсходованного интернет-траффика за месяц всеми абонентами.

In [ ]:
# для абонентов тарифа smart. По всем пользователям


In [ ]:
# для абонентов тарифа ultra. По всем пользователям


По описательной статистике можем сделать вывод о том, что в среднем за месяц абоненты тарифа smart расходуют примерно в два раза больше минут разговоров, отправляют сообщения и расходуют интернет-траффик. Среднемесячная выручка от этих абонентов примерно в 1,5 раза выше выручки, полученной с абонентов тарифа ultra.

Сравнение максимальных значений с 75% квантилью и минимальных значений с 25% квантилью показывает, что данные не содержат большого числа выбросов, они достаточно редки.

Медианные и средние значения принимают достаточно близкие значения, т.е. распределения почти симметричны.

Оценим средние показатели. Вычислим основные числовые характеристики случайных величин числа израсходованных минут разговоров, отправленных сообщений и израсходованного интернет-траффика за месяц на одного абонента.

In [ ]:
# для абонентов тарифа smart. На одного абонента


In [ ]:
# для абонентов тарифа ultra. На одного абонента


По описательной статистике можем сделать вывод о том, что в среднем за месяц абоненты тарифа smart расходуют примерно в два раза больше минут разговоров, отправляют сообщения и расходуют интернет-траффик. Среднемесячная выручка от этих абонентов примерно в 2 раза выше выручки, полученной с абонентов тарифа ultra.

Сравнение максимальных значений с 75% квантилью и минимальных значений с 25% квантилью показывает, что данные не содержат большого числа выбросов, они достаточно редки.

Медианные и средние значения принимают достаточно близкие значения, т.е. распределения почти симметричны.

Величина среднеквадратического отклонения характеризует степень разброса данных относительно центра. Согласно правилу трёх сигм в интервале шириной 6 сигм с центром в точке мат. ожидания лежит 99,73% всех значений. Сравним случайные величины величины по характеристике среднеквадратического отклонения.

Для количества потраченных минут на разговоры для тарифов smart и ultra ср.кв.отклонения равны 72,1 и 77 минут соответственно. Это означает, что 95% абонентов тарифного плана smart в месяц разговаривают примерно от 386 - 2`*`72 до 386 + 2`*`72, т.е. тратят на разговоры от 242 до 530 минут. Небольшая часть абонентов превышает тарифную планку, а, значит, выручка по этой части будет небольшая. Для 95% абонентов тарифного плана ultra интервал использования минут разговоров в месяц составляет примерно от 341 до 649 минут. Видим, что абоненты тарифного плана ultra тратят на разговоря больше минут, но не выходят за пределы лимита по этому тарифу.

Для количества отправленных сообщений для тарифов smart и ultra ср.кв.отклонения равны 5 и 12 сообщений соответственно. Это означает, что 95% абонентов тарифного плана smart в месяц отправляют примерно от 31 - 2`*`5 до 31 + 2`*`5, т.е. отправляют от 21 до 41 сообщения в месяц. За пределы лимита по тарифу не выходят, дополнительная выручка по этой части будет практически отсутствовать. Для 95% абонентов тарифного плана ultra интервал использования минут разговоров в месяц шире и составляет примерно от 44- 24 до 44 + 24 сообщений, т.е. от 20 до 68. Разброс примерно в 2 раза больше, чем для абонентов тарифного плана smart. Видим, что абоненты тарифного плана ultra по числу отправленных сообщений не выходят за пределы лимита по этому тарифу.

Для объема использованного интернет-траффика для тарифов smart и ultra ср.кв.отклонения равны 2666 и 2969 Мб, но при округлении оператором получаем одинаковые 3 Гб. Однако средние значения разные, поэтому абоненты тарифа smart примерно раходуют от 9 до 21 Гб и примерно 50% этих абонентов превышают лимит. Поэтому оплачивают дополнительный объем интернет-траффика и формируют дополнительную выручку. Для абонентов тарифа ultra объем интернет-траффика составляет примерно от 18 - 6 до 18 + 6 Гб, т.е. от 12 до 24 Гб, что немного больше, чем для абонентов smart. Однако никто из абонентов не выходит за пределы тарифного плана и выручка для них формируется исключительно за счет абонентской платы.

Изучим распределения числа использованных минут разговоров, числа отправленных сообщений, объема интернет-траффика, которые расходуют абоненты каждого тарифа.



In [ ]:
for x in ['minute_per_month', 'messages_per_month', 'gb_per_month','income']:
  plt.figure(figsize = (9,6))
  sns.histplot()
  sns.histplot()

  plt.xlabel(x)
  plt.ylabel('Frequency')
  plt.grid()
  plt.legend(loc = 'best')
  plt.show()

Распределение количества потраченных минут на разговоры для тарифа smart и ultra похожи на нормальное распределение. Для тарифа smart это распределение более симметрично. Мат.ожидание и медиана имеют примерно одинаковые значения. Основная часть абонентов расходует от 200 до 500 минут разговоров. Для тарифа ultra толстый хвост слева, асимметрия влево, т.е. основная часть абонентов расходует от 0 до 750 минут разговоров.

Распределение количества отправленных сообщений для каждого тарифа похоже на экспоненциальное распределение. Большая часть абонентов отправляет небольшое количество сообщений. Абоненты тарифа smart отправляют в среднем около 30 сообщений, абоненты тарифа ultra - в среднем около 40 сообщений.

Распределение объема израсходованного интернет-траффика похоже на нормальное, симметричное для абонентов каждого из тафных планов. Центр для абонентов smart примерно на 15 Гб, центр распределения для абонентов ultra - примерно на 19 Гб.

Гистограммы выручки имеют одинаковую форму, но смещены относительно друг друга с учетом абонентской платы. Большинство абонентов тарифа smart приносят выручку от 550 до 1200 рублей, большинство абонентов тарифа ultra приносят выручку от 1950 до 2200 рублей. Тонкие длинные хвосты справа.

Вычислите коэффициенты корреляции показателей для абонентов тарифного плана smart и ultra. От каких показателей в большей степени формируется прибыль компании.

In [ ]:
# сформируем список столбцов для вычисления коэффициентов корреляции
num_col = abonents.select_dtypes(include='number').columns
num_col

In [ ]:
plt.figure(figsize = (9,6))
corr_smart =
sns.heatmap()
plt.show()

In [ ]:
plt.figure(figsize = (9,6))
corr_ultra =
sns.heatmap()
plt.show()

# Шаг 4. Проверьте гипотезы

•	средняя выручка пользователей тарифов «Ультра» и «Смарт» различаются;

•	средняя выручка пользователи из Москвы отличается от выручки пользователей из других регионов.

Пороговое значение alpha задайте самостоятельно.

Поясните:

•	как вы формулировали нулевую и альтернативную гипотезы;

•	какой критерий использовали для проверки гипотез и почему.

###  1. Средняя выручка пользователей тарифов «Ультра» и «Смарт» различаются;

Проверьте гипотезу о равенстве средних, используя `st.ttest_ind`.

Пусть mean(income_{smart}), mean(income_{ultra}) - математические ожидания выручки абонентов тарифа smart и ultra соответственно.

Основная гипотеза $$H_0: mean(income_{smart}) = mean(income_{ultra}) $$

Альтернативная гипотеза: $$H_1: mean(income_{smart}) \neq mean(income_{ultra})$$ (знак <> следует понимать как не равно). Будем строить двустороннюю критическую область.

Выберем уровень значимости `alpha` = 0.05 (ошибка первого рода, т.е. вероятность отклонить нулевую гипотезу, если она верна).

Для проверки гипотезы о равенстве средних применяют `t`-статистику Стьюдента.



Для вычисления наблюдаемого значения критерия вычислите средние и дисперсии выручки абонентов тарифа smart и ultra.


In [ ]:
# Выделим массивы, содержащие выручку от абонентов разных тарифных планов
mean_income_smart = smart_abonent['income'].values
mean_income_ultra = ultra_abonent['income'].values

In [ ]:
# Оценим дисперсии этих совокупностей для проверки гипотезы
print(np.var(mean_income_smart).round(2))
print(np.var(mean_income_ultra).round(2))

Если дисперсии совокупностей различны, при использовании `st.ttest_ind` используйте параметр `equal_var = False`.

In [ ]:
alpha = 0.05
result = st.ttest_ind()
print('p-значение',result.pvalue.round(2))
if (result.pvalue < alpha):
  print('отвергаем нулевую гипотезу')
else:
  print('нет оснований отвергнуть нулевую гипотезу')

Основная гипотеза отвергается в пользу альтернативной. Делаем вывод, что средние выручки абонентов тарифа smart и ultra различаются существенно.

### 2. Средняя выручка пользователи из Москвы отличается от выручки пользователей из других регионов

Проверим гипотезу о равенстве средних.

Пусть mean(income_{moscow}), mean(income_{other}) - математические ожидания выручки абонентов Москвы и других регионов соответственно.

Основная гипотеза $$H_0: mean(income_{moscow}) = mean(income_{other})$$

Альтернативная гипотеза: $$H_1: mean(income_{moscow}) \neq mean(income_{other})$$
Будем строить двустороннюю критическую область.

Выберем уровень значимости `alpha` = 0.05 (ошибка первого рода, т.е. вероятность отклонить нулевую гипотезу, если она верна).

Для проверки гипотезы о равенстве средних применяют `t`-статистику Стьюдента.

Для вычисления наблюдаемого значения критерия вычислим средние и дисперсии выручки абонентов пользователей Москвы и других регионов

Сформируем массивы с выручкой абонентов Москвы и других регионов. Обратимcя к общей таблице `abonents`.


In [ ]:
mean_income_moscow =
mean_income_other =

In [ ]:
# Оценим дисперсии этих совокупностей для проверки гипотезы


Дисперсии совокупностей различны, поэтому при использовании `st.ttest_ind` параметр `equal_var = False`.

Таким образом, средняя выручка пользователи из Москвы не отличается от выручки пользователей из других регионов

# Шаг 5. Напишите общий вывод

Сделайте вывод, какой тариф следует развивать компании. Пользователи какого тарифа приносят больше прибыли и за счет каких услуг.
